# 3회차 · 문서 처리와 내 질문 검색

셀 1~9에서는 약관 세 조각으로 상태·분기를 연습합니다. 셀 5의 라우터를 채운 뒤 다음 셀로 진행하세요. 마지막 개인 실습은 전체 인덱스와 BGE-M3가 필요합니다.

In [1]:
# [셀 1] 준비
import sys
from importlib.metadata import version

print("python        ", sys.version.split()[0])
for pkg in ("langgraph", "langchain-core"):
    print(f"{pkg:14}", version(pkg))

python         3.12.10
langgraph      1.2.24
langchain-core 1.8.4


## 1. State — 노드들이 주고받는 하나의 딕셔너리

그래프는 **상태 하나**를 노드들이 차례로 고쳐 나가는 구조입니다. 상태는 딕셔너리이고, 어떤 키가 있는지를 `TypedDict` 로 적어 둡니다.

- `total=False` 를 붙이는 이유: 처음부터 모든 키가 있지는 않습니다. `answer` 는 마지막 노드가 채웁니다.
- 여기 적히지 않은 키는 노드가 반환해도 **버려집니다.** 오늘 코드의 `IngestState` 도 같은 규칙입니다.

In [3]:
# [셀 2] State
from typing import Literal, TypedDict


class State(TypedDict, total=False):
    question: str        # 사용자가 물은 것
    hits: list[dict]     # 검색된 조항들
    grade: str           # 근거가 충분한가: sufficient / none
    answer: str          # 최종 답변
    trace: list[str]     # 어느 노드를 지나왔는가. 경로 자체가 기록입니다


print("State 의 키:", list(State.__annotations__))

State 의 키: ['question', 'hits', 'grade', 'answer', 'trace']


## 작은 검색기로 분기 연습하기

약관 세 조각에서 두 글자 단위로 겹치는 표현을 찾습니다. 실제 Dense 검색은 뒤의 개인 실습에서 사용합니다.

In [4]:
# [셀 3] 장난감 검색기
CORPUS = [
    {"doc_id": "toy_deposit", "article": "제10조(중도해지이율)",
     "text": "만기일 이전에 해지하는 경우 가입일 당시 영업점에 게시한 중도해지이율을 적용합니다."},
    {"doc_id": "toy_deposit", "article": "제3조(가입대상)",
     "text": "실명의 개인으로 하며 1인 1계좌에 한하여 가입할 수 있습니다."},
    {"doc_id": "toy_loan", "article": "제7조(중도상환수수료)",
     "text": "대출일부터 3년 이내에 상환하는 경우 상환금액에 수수료율을 곱한 금액을 부과합니다."},
]

MIN_OVERLAP = 2      # 겹치는 글자쌍이 이보다 적으면 검색 결과에서 뺍니다


def bigrams(s: str) -> set[str]:
    s = "".join(s.split())
    return {s[i:i + 2] for i in range(len(s) - 1)}


def search(query: str, k: int = 2) -> list[dict]:
    scored = [(len(bigrams(query) & bigrams(c["article"] + c["text"])), c) for c in CORPUS]
    scored.sort(key=lambda t: -t[0])
    return [c for n, c in scored if n >= MIN_OVERLAP][:k]


for q in ["중도해지이율은 어떻게 적용되나요?", "암 진단비는 얼마인가요?"]:
    print(q, "→", [c["article"] for c in search(q)] or "검색 결과 없음")

중도해지이율은 어떻게 적용되나요? → ['제10조(중도해지이율)']
암 진단비는 얼마인가요? → 검색 결과 없음


## 3. Node — 상태를 받아 바뀐 부분만 반환하는 함수

노드는 특별한 클래스가 아니라 **그냥 함수**입니다. 규칙은 두 개뿐입니다.

1. 인자로 상태를 받습니다.
2. 바뀐 키만 담은 딕셔너리를 반환합니다. LangGraph 가 원래 상태에 합쳐 줍니다.

`trace` 를 `state.get("trace", []) + [...]` 로 이어 붙이는 것을 봐 두세요. 오늘 코드의 `warnings` 도 같은 방식으로 쌓입니다.

In [5]:
# [셀 4] 노드 네 개
def retrieve(state: State) -> State:
    hits = search(state["question"])
    return {"hits": hits, "trace": state.get("trace", []) + ["retrieve"]}


def grade(state: State) -> State:
    """근거가 쓸 만한가. 오늘 코드에서는 이 자리에 검증 게이트가 들어갑니다."""
    verdict = "sufficient" if state.get("hits") else "none"
    return {"grade": verdict, "trace": state.get("trace", []) + ["grade"]}


def answer(state: State) -> State:
    h = state["hits"][0]
    return {"answer": f"{h['article']}에 따르면, {h['text']}  [근거] {h['doc_id']} · {h['article']}",
            "trace": state.get("trace", []) + ["answer"]}


def abstain(state: State) -> State:
    """근거가 없으면 답을 만들지 않습니다. 이 수업에서 가장 중요한 노드입니다."""
    return {"answer": "제공된 문서에서 근거를 찾지 못했습니다. 다른 표현으로 물어보시겠어요?",
            "trace": state.get("trace", []) + ["abstain"]}


print(grade({"hits": [CORPUS[0]]}))
print(grade({"hits": []}))

{'grade': 'sufficient', 'trace': ['grade']}
{'grade': 'none', 'trace': ['grade']}


## 4. 조건부 엣지 — 실습 1

`retrieve → grade` 는 항상 같은 방향이라 보통 엣지로 잇습니다. `grade` 다음은 상태에 따라 갈립니다. 이때 쓰는 것이 **라우터 함수**입니다.

라우터는 노드가 아닙니다. 상태를 **읽기만** 하고 다음 목적지의 **이름(문자열)** 을 반환합니다. 반환형에 `Literal` 을 적어 두면 목적지 이름에 오타가 나도 편집기가 잡아 줍니다.

**실습 1.** 아래 `route_after_grade` 를 채웁니다. `grade` 가 `"sufficient"` 면 `"answer"`, 아니면 `"abstain"` 을 반환합니다. 한 줄이면 됩니다. 셀을 실행해서 "실습 1 통과"가 나오면 끝입니다.

오늘 코드의 `nodes.route_after_gate` 가 정확히 이 모양입니다. 거기서는 `state["route"]` 를 그대로 반환합니다.

In [6]:
# [셀 5] 실습 1 — 라우터 함수
def route_after_grade(state: State) -> Literal["answer", "abstain"]:
    # ── 실습 1: 여기를 채우세요 ──
    # grade 가 "sufficient" 면 "answer", 아니면 "abstain"
    return "answer" if state.get("grade") == "sufficient" else "abstain"


assert route_after_grade({"grade": "sufficient"}) == "answer", "근거가 충분하면 answer 로 보내야 합니다"
assert route_after_grade({"grade": "none"}) == "abstain", "근거가 없으면 abstain 으로 보내야 합니다"
assert route_after_grade({}) == "abstain", "grade 가 아예 없어도 abstain 입니다"
print("실습 1 통과")

실습 1 통과


## 5. 조립하고 그림 보기

노드를 등록하고, 시작점을 정하고, 엣지를 잇고, `compile()` 합니다. 조건부 엣지는 `add_conditional_edges(노드, 라우터, {라우터가 돌려주는 값: 다음 노드})` 입니다.

`compile()` 한 그래프는 mermaid 텍스트로 그림을 그려 줍니다. 그림을 손으로 그리지 않아서 코드와 그림이 어긋나지 않습니다.

In [7]:
# [셀 6] 조립
from langgraph.graph import END, StateGraph


def build():
    g = StateGraph(State)
    g.add_node("retrieve", retrieve)
    g.add_node("grade", grade)
    g.add_node("answer", answer)
    g.add_node("abstain", abstain)

    g.set_entry_point("retrieve")
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", route_after_grade,
                            {"answer": "answer", "abstain": "abstain"})
    g.add_edge("answer", END)
    g.add_edge("abstain", END)
    return g.compile()


graph = build()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade(grade)
	answer(answer)
	abstain(abstain)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade -.-> abstain;
	grade -.-> answer;
	retrieve --> grade;
	abstain --> __end__;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 두 질문의 방문 순서 확인하기

각 질문에서 마지막으로 실행될 노드를 예상하고 셀 7의 trace를 읽으세요. 예제는 방문 순서를 직접 기록합니다. 전체 문서 처리 리포트는 문서별 최종 상태와 판정 건수를 집계합니다.

In [8]:
# [셀 7] 실행
for q in ["중도해지이율은 어떻게 적용되나요?", "암 진단비는 얼마인가요?"]:
    out = graph.invoke({"question": q})
    print("Q.", q)
    print("   경로:", " → ".join(out["trace"]))
    print("   답 :", out["answer"][:60])
    print()

Q. 중도해지이율은 어떻게 적용되나요?
   경로: retrieve → grade → answer
   답 : 제10조(중도해지이율)에 따르면, 만기일 이전에 해지하는 경우 가입일 당시 영업점에 게시한 중도해지이율을 적

Q. 암 진단비는 얼마인가요?
   경로: retrieve → grade → abstain
   답 : 제공된 문서에서 근거를 찾지 못했습니다. 다른 표현으로 물어보시겠어요?



## 7. stream — 노드마다 무엇이 바뀌었는지 본다

`invoke` 는 최종 상태만 줍니다. `stream` 은 노드가 끝날 때마다 **그 노드가 돌려준 부분 딕셔너리**를 흘려 줍니다. 노드가 "바뀐 키만" 반환한다는 규칙이 여기서 눈에 보입니다.

In [9]:
# [셀 8] stream
for step in graph.stream({"question": "중도상환수수료는 언제 내나요?"}):
    for node, patch in step.items():
        print(f"[{node:9}] 바뀐 키: {sorted(patch)}")

[retrieve ] 바뀐 키: ['hits', 'trace']
[grade    ] 바뀐 키: ['grade', 'trace']
[answer   ] 바뀐 키: ['answer', 'trace']


## 8. 오늘 코드에서는 어디에 있나

| 이 노트북 | 오늘 코드 (`src/finrag/ingestion/`) |
|---|---|
| `State` | `nodes.IngestState` — doc_id, pages, report, route, chunks … |
| 노드 함수 `retrieve`, `grade` … | `nodes.classify`, `parse`, `gate`(gate 실습), `reparse`, `ocr`, `llm_repair`, `chunk`, `dead_letter` |
| 라우터 `route_after_grade`(실습 1) | `nodes.route_after_gate`, `route_after_reparse`, `route_after_ocr` |
| `build()` 의 엣지 연결 | `graph.build_ingest_graph`(그래프 연결 실습) |
| `graph.invoke({"question": …})` | `pipelines/ingest.py` 가 문서마다 `invoke({"doc_id": …, "path": …})` |

아래 셀은 오늘 코드의 그래프 그림입니다. 그래프 연결 실습 을 채우기 전에는 "아직 채우지 않았습니다"가 나오고, 채우고 나서 다시 실행하면 그림이 나옵니다.

In [12]:
# [셀 9] 인제스천 연결 관계 (그래프 연결 실습 뒤에 다시 실행)
import importlib
import sys
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo / "src"))

# 앞서 미작성 함수를 import했어도 저장한 파일을 새로 읽습니다.
import finrag.ingestion.nodes as ingest_nodes
import finrag.ingestion.graph as ingest_graph
importlib.invalidate_caches()
importlib.reload(ingest_nodes)
importlib.reload(ingest_graph)

try:
    print(ingest_graph.build_ingest_graph().get_graph().draw_mermaid())
except NotImplementedError as e:
    print("아직 채우지 않았습니다:", e)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	parse(parse)
	gate(gate)
	reparse(reparse)
	ocr(ocr)
	llm_repair(llm_repair)
	chunk(chunk)
	dead_letter(dead_letter)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify --> parse;
	gate -. &nbsp;pass&nbsp; .-> chunk;
	gate -. &nbsp;fail&nbsp; .-> dead_letter;
	gate -.-> ocr;
	gate -.-> reparse;
	llm_repair -.-> chunk;
	llm_repair -. &nbsp;fail&nbsp; .-> dead_letter;
	ocr -.-> chunk;
	ocr -. &nbsp;fail&nbsp; .-> dead_letter;
	parse --> gate;
	reparse -. &nbsp;pass&nbsp; .-> chunk;
	reparse -. &nbsp;repair&nbsp; .-> llm_repair;
	reparse -.-> ocr;
	chunk --> __end__;
	dead_letter --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 내 질문 검색

앞의 터미널 실행이 끝난 뒤 진행하세요. 개인 결과만 저장합니다. 첫 정답 순위는 지정한 근거 중 가장 앞선 순위이며, 모든 근거의 순위도 함께 남깁니다.

In [55]:
# [셀 10] 질문과 정답 후보 불러오기
import json, os, sys, time
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "pipelines/baseline.py").is_file():
    raise RuntimeError("현재 회차 포크 또는 그 안의 notebooks 폴더에서 실행하세요.")
os.environ.setdefault("HF_HOME", str(root / "models"))
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "src"))
def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))
def norm(s):
    return "".join(s.split())
records = read_jsonl(root / "data/chunks/chunks.jsonl")
lookup = {c["chunk_id"]: c for c in records}
def sample():
    seed = next(x for x in read_jsonl(root / "data/golden/seed_golden.jsonl") if x["id"] == "S01")
    cid = "hanacard_std_2017#제31조#1"
    return {"id": "S01", "question": seed["question"], "source": "공통 예시 S01 · 제31조만 확인",
            "gold_chunk_ids": [cid], "quotes": {cid: "다음 달 결제월에 잔여결제금액과"}}

# 내 파일에서 사용할 문항의 번호 (0은 첫 번째)
choice = 1
path = root / "data/golden/student_q.jsonl"
if path.is_file():
    choices = read_jsonl(path)
    for i, q in enumerate(choices):
        print(i, q["question"])
    if not 0 <= choice < len(choices):
        raise ValueError("위 목록에서 사용할 문항 번호를 choice에 적으세요.")
    item = dict(choices[choice], source="개인 질문", quotes={})
else:
    item = sample()
    print("개인 파일이 없어 공통 예시 S01을 사용합니다.")

print(item["source"], item["question"])
for cid in item["gold_chunk_ids"]:
    print(cid, lookup[cid]["text"][:800] if cid in lookup else "현재 청크에 없음 · 아래 원문 검색 사용")

0 예금거래기본약관 제9조의2에 따르면 거치식·적립식 예금은 언제 휴면예금으로 보나요?
1 카카오뱅크 정기예금은 만기 전에 일부만 긴급출금할 수 있나요? 몇 번까지, 어떤 조건으로 가능한가요?
2 카카오뱅크 정기예금 특약의 2024년 10월 17일 시행 개정 내용은 2023년에 가입한 기존 고객에게도 적용되나요?
개인 질문 카카오뱅크 정기예금은 만기 전에 일부만 긴급출금할 수 있나요? 몇 번까지, 어떤 조건으로 가능한가요?
kakao_deposit_terms_2024#제10조#1 제10 조 긴급출금 
이 예금의 긴급출금은 가입일(또는 자동연장일)로부터 만기일 전일까지 최대 2 회까지 가능합니다. 다만, 긴급출금 후 최소 유지금액은 100 만원 이상이어
야 하며 긴급출금 된 금액에 대해서는 중도해지금리를 적용합니다.


In [56]:
# [셀 11] 청크 원문 찾기
# 원문이 뒤쪽에 있거나 ID가 바뀌었다면 값을 입력한 뒤 이 셀을 실행하세요.
doc_id = ""      # 예: hanacard_std_2017
phrase = ""      # 예: 다음 달 결제월에
if phrase.strip():
    matches = [c for c in records if (not doc_id or c["doc_id"] == doc_id)
               and norm(phrase) in norm(c["text"])]
    print("일치한 청크 수", len(matches))
    for c in matches[:20]:
        print(c["chunk_id"], "쪽", c.get("page_start"), "조항", c.get("article"))

In [47]:
# [셀 12] 근거 구절과 ID 확인
# 개인 질문은 아래 사전에 실제로 확인한 현재 ID와 근거 구절을 넣으세요.
# 예: {"hanacard_std_2017#제31조#1": "다음 달 결제월에 잔여결제금액과"}
# 여러 청크가 필요하면 각각을 적습니다. 4회차에서는 3회차 기록을 자동으로 불러옵니다.
confirmed = dict(item.get("quotes", {}))
confirmed = {"kakao_deposit_terms_2024#제10조#1": "최대 2회까지 가능합니다"}
if not confirmed or any(not quote.strip() or cid not in lookup or norm(quote) not in norm(lookup[cid]["text"])
                        for cid, quote in confirmed.items()):
    raise ValueError("근거를 확인하세요. 바로 위 셀에서 문서와 구절을 찾아 confirmed에 적으세요.")
for cid, quote in confirmed.items():
    print(cid, quote)

kakao_deposit_terms_2024#제10조#1 최대 2회까지 가능합니다


In [57]:
# [셀 13] Dense 검색과 첫 순위
from pipelines.baseline import build as build_dense
from finrag.index import qdrant
def measure(run):
    start = time.perf_counter()
    try:
        out = run(item["question"])
    finally:
        qdrant.close()
    ids = out["chunk_ids"]
    positions = {cid: ids.index(cid) + 1 if cid in ids else None for cid in confirmed}
    first = min((i for i in positions.values() if i is not None), default=None)
    result = {"first_rank": first, "ranks_by_gold": positions, "top_ids": ids,
              "elapsed_ms": round(1000 * (time.perf_counter() - start), 1),
              "reranked": out.get("rerank", {}).get("reranked")}
    print("첫 정답", first if first is not None else f"상위 {len(ids)}개에 없음", " / 경과(ms)", result["elapsed_ms"])
    for i, cid in enumerate(ids[:5], 1):
        print(i, cid, lookup.get(cid, {}).get("text", "")[:80].replace("\n", " "))
    if result["reranked"] is not None:
        print("재정렬 실행:", result["reranked"])
    return result
runs = {}
print("Dense")
runs["Dense"] = measure(build_dense(k=10))

Dense
첫 정답 2  / 경과(ms) 769.9
1 kakao_deposit_2026_txt#본문#9 액만 출금 가능                  • 신청기간 : 가입일(또는 자동연장일)로부터 만기일 전일까지     긴급출금           
2 kakao_deposit_terms_2024#제10조#1 제10 조 긴급출금  이 예금의 긴급출금은 가입일(또는 자동연장일)로부터 만기일 전일까지 최대 2 회까지 가능합니다. 다만, 긴급출금 후 최소 
3 kakao_deposit_special_2019#본문#1 카카오뱅크 특별판매 정기예금 상품설명서 (요약) 이 설명서는 은행이용자의 상품에 대한 이해를 돕기 위한 참고자료이며, 실제 계약은 「카카오뱅크 
4 kakao_deposit_2026_txt#본문#3 회까지 연장 가능   긴급출금 (부분해지)         가능                                    만기일 전에 납입금
5 kakao_deposit_2026_pdf#본문#2 확인하시고 궁금하신 부분은 고객센터로 문의하시기 바랍니다. 중도해지  만기일 전에 중도에 해지하는 경우 기본금리보다 낮은 중도해지금리가 적용 (


In [54]:
# [셀 14] 개인 결과 저장
record = {"id": item["id"], "source": item["source"], "question": item["question"],
          "gold_chunk_ids": list(confirmed), "quotes": confirmed, "chunk_count": len(records), "runs": runs}

target = root / "results/my_question_s03.json"
target.parent.mkdir(exist_ok=True)
target.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print(target)

c:\subin\finance-agentic-rag-s03\results\my_question_s03.json
